In [8]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
#from Functions.AutoCloud_V4_4 import *
from Functions.AutoCloud_V4_6 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
from Functions.TedaOptimize_V4_6 import *     # Optimize, OptimizeSeq, BestParams, SimSeq
import copy

FileName = ipynbname.name()

RS = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaOpt_R5_T1_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(PARAMS_PATH).iloc[:,:]
EOL = 0.3

In [5]:
df

,WAPE_mean,m,nI,nR,nO,mO,N1,N2,N3,TAU,activation,w,WAPE_RUL_B1_1,WAPE_HI_B1_1,WAPE_RUL_B1_2,WAPE_HI_B1_2
0,0.038627,1.7500,26,[40],5,0,1,7,5,22,0,10,0.197688,0.012283,0.230492,0.009575
1,0.037482,2.0250,17,[57],2,2,1,4,9,19,0,14,0.283294,0.007871,0.258199,0.008784
2,0.034758,1.9500,38,[71],2,25,2,8,1,3,0,4,0.288705,0.011835,0.291650,0.002586
3,0.030109,2.0250,20,[44],8,1,1,6,3,31,0,15,0.201636,0.009745,0.243577,0.006766
4,0.025167,1.8875,23,[54],6,17,1,2,1,17,0,9,0.201059,0.004589,0.226359,0.009265
5,0.023472,1.7875,39,[54],6,39,1,8,9,25,0,8,0.118445,0.008489,0.205558,0.008811
6,0.022508,1.9625,32,[48],6,26,1,2,4,2,0,9,0.247760,0.003741,0.295157,0.006010
7,0.020814,2.0250,9,[38],6,9,1,2,6,17,0,7,0.232710,0.008046,0.192226,0.003253
8,0.019561,2.0000,29,[68],3,23,1,1,6,19,0,10,0.230905,0.006553,0.245700,0.003395
9,0.017512,1.9250,17,[43],9,3,1,3,7,27,0,20,0.203240,0.007542,0.266652,0.002132


In [ ]:
i = 25
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
teda.HI_Granularity()
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


In [15]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
i = 25
sig = HI['PC1'].values
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


In [ ]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B1_4','B1_5','B2_1','B2_2','B2_4',][:2]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]
#DSIs = [DSI.drop(["FTF"], axis=1) for DSI in DSIs]
ftrs = len(DSIs[0].columns)
DSIs = [RS1,RS2]
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )
w = 1
for DSI in DSIs: 
    teda.SimGranulation(WindowDSI(DSI[19:],w))

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=ftrs,dsi_mode='hide')

In [ ]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')

sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaRNN_Opt_R3_Seq_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X1, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X1,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X2, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X2,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [ ]:
wR = (0.101690 - 0)/(0.3 - 0) 
wH = (0.013605	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

wR = (0.575803	 - 0)/(0.3 - 0) 
wH = (0.003675	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

In [ ]:
wR = (0.226344 - 0)/(0.3 - 0) 
wH = (0.001195		 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

wR = (0.224894 - 0)/(0.3 - 0) 
wH = (0.000693	 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

In [ ]:
path = r'Optimization\TedaOpt_R5_T0_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(path)
df = df.sort_values(df.columns[0], ascending=True)
df